# Digit Recognizer — CNN (MNIST)

**目标**：识别 28×28 手写数字（0–9）  
**指标**：Accuracy

## 预处理
- 像素归一化到 `[0,1]`，形状 `[N,1,28,28]`  
- 10% 训练作为验证集

## 模型
- **2×Conv(32,64) + BN + ReLU + MaxPool + Dropout → FC(256→10)**  
- **数据增强**：轻微旋转（±10°）与平移  
- **优化**：Adam + **CosineAnnealingLR**，**早停**  
- **TTA**：0°/±10° 固定旋转投票

## 结果
| 指标 | 值 |
|---|---|
| 验证集 Accuracy | **≈ 0.99±** |
| Public LB | **0.99307** |

## 复现 / 提交
生成 `submission_mnist_v2.csv` 并提交。  
**依赖**：`torch`, `torchvision`, `numpy`, `pandas`（可选 GPU）


In [ ]:
# ===== Digit Recognizer v1: 简洁CNN基线（PyTorch） =====
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from pathlib import Path

data_dir = Path("/kaggle/input/digit-recognizer")
train = pd.read_csv(data_dir/"train.csv")
test  = pd.read_csv(data_dir/"test.csv")

X = train.drop(columns=["label"]).values.astype(np.float32).reshape(-1,1,28,28)/255.0
y = train["label"].values.astype(np.int64)
X_test = test.values.astype(np.float32).reshape(-1,1,28,28)/255.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
n_val = int(len(ds)*0.1); n_train = len(ds) - n_val
train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,padding=1)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        self.drop  = nn.Dropout(0.25)
        self.fc1   = nn.Linear(64*14*14,128)
        self.fc2   = nn.Linear(128,10)
    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = F.relu(self.conv2(x))
        x = self.drop(x).view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)

model = Net().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

def acc(dl):
    model.eval(); correct=total=0
    with torch.no_grad():
        for xb,yb in dl:
            xb,yb=xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred==yb).sum().item(); total += yb.size(0)
    return correct/total

best_acc=0.0
for epoch in range(8):
    model.train()
    for xb,yb in train_dl:
        xb,yb=xb.to(device), yb.to(device)
        opt.zero_grad(); loss=crit(model(xb), yb); loss.backward(); opt.step()
    va = acc(val_dl)
    print(f"Epoch {epoch+1}: val_acc={va:.4f}")
    if va>best_acc: best_acc=va

# 预测与提交
model.eval()
test_tensor = torch.from_numpy(X_test).to(device)
pred = []
with torch.no_grad():
    for i in range(0, len(test_tensor), 512):
        pred.append(model(test_tensor[i:i+512]).argmax(1).cpu().numpy())
pred = np.concatenate(pred)
sub = pd.DataFrame({"ImageId": np.arange(1, len(pred)+1), "Label": pred})
sub.to_csv("submission_mnist_v1.csv", index=False)
print("Saved submission_mnist_v1.csv")


In [ ]:
# 优化版PyTorch
# ===== Digit Recognizer v2: Augmentation + Cosine LR + Early Stop + Light TTA =====
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as T
from pathlib import Path

# reproducibility
torch.manual_seed(42); np.random.seed(42)

# 1) 读取数据
data_dir = Path("/kaggle/input/digit-recognizer")
train = pd.read_csv(data_dir/"train.csv")
test  = pd.read_csv(data_dir/"test.csv")

X = train.drop(columns=["label"]).values.astype("float32").reshape(-1,1,28,28) / 255.0
y = train["label"].values.astype("int64")
X_test = test.values.astype("float32").reshape(-1,1,28,28) / 255.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) 数据集（训练集做轻量增强）
class AugDS(Dataset):
    def __init__(self, X, y=None, aug=False):
        self.X = torch.from_numpy(X)           # [N,1,28,28] float32 tensor
        self.y = None if y is None else torch.from_numpy(y)
        self.aug = aug
        self.tf = T.Compose([
            T.RandomRotation(10, fill=0),                  # 随机 [-10°, +10°]
            T.RandomAffine(0, translate=(0.08,0.08), fill=0)  # 轻微平移
        ])
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        img = self.X[i]
        if self.aug:
            img = self.tf(img)
        return (img, self.y[i]) if self.y is not None else img

ds_all = AugDS(X, y, aug=True)
n_val = int(len(ds_all)*0.1); n_tr = len(ds_all) - n_val
tr_ds, va_ds = random_split(ds_all, [n_tr, n_val], generator=torch.Generator().manual_seed(42))

tr_dl = DataLoader(tr_ds, batch_size=256, shuffle=True, num_workers=2)
va_dl = DataLoader(va_ds, batch_size=512, shuffle=False, num_workers=2)

# 3) 简洁 CNN（带 BN + Dropout）
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3, padding=1); self.b1 = nn.BatchNorm2d(32)
        self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.b2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2,2); self.drop = nn.Dropout(0.30)
        self.fc1 = nn.Linear(64*14*14, 256); self.b3 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        x = self.pool(F.relu(self.b1(self.c1(x))))
        x = F.relu(self.b2(self.c2(x)))
        x = self.drop(x).view(x.size(0), -1)
        x = F.relu(self.b3(self.fc1(x)))
        x = self.drop(x)
        return self.fc2(x)

model = Net().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12)
crit = nn.CrossEntropyLoss()

# 4) 训练（简易早停）
best_acc, patience, bad = 0.0, 4, 0
for epoch in range(12):
    model.train()
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward(); opt.step()
    # 验证
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in va_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred==yb).sum().item(); total += yb.size(0)
    acc = correct/total
    sched.step()
    print(f"Epoch {epoch+1}: val_acc={acc:.4f}")
    if acc > best_acc:
        best_acc, bad = acc, 0
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience: 
            print("Early stopping.")
            break

# 恢复最佳模型
if 'best_state' in locals():
    model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
model.eval()

# 5) 轻量 TTA（固定 0°, +10°, -10°）
ds_te = AugDS(X_test, None, aug=False)
angles = [0, 10, -10]
tts = [T.RandomRotation((a, a), fill=0) for a in angles]  # 用 (a,a) 固定角度

with torch.no_grad():
    logits = 0
    for tfm in tts:
        # 对每一张测试图片应用一个固定角度的旋转，再批量预测
        tmp = torch.stack([tfm(img) for img in ds_te])         # [N,1,28,28]
        part = []
        for i in range(0, len(tmp), 512):
            part.append(model(tmp[i:i+512].to(device)).softmax(dim=1).cpu())
        logits += torch.cat(part, dim=0)

pred = logits.argmax(dim=1).numpy()

# 6) 生成提交
sub = pd.DataFrame({"ImageId": np.arange(1, len(pred)+1), "Label": pred})
sub.to_csv("submission_mnist_v2.csv", index=False)
print("Saved submission_mnist_v2.csv")

